# Project 13: RL for Portfolio Risk Management

Reinforcement learning agents for dynamic portfolio allocation with explicit
tail-risk constraints. Pure numpy implementation using Natural Evolution
Strategies (NES).

**Sections:**
1. Generate synthetic multi-asset returns with regime change
2. Define benchmarks: equal weight, mean-variance, risk parity
3. Train RL agent with CVaR constraint, show loss curve
4. Backtest all strategies, compare metrics table
5. Visualizations: cumulative returns, allocation evolution, risk-return scatter, drawdowns
6. Interpretation: when does RL add value?

In [ ]:
import sys
from pathlib import Path

# Add project source to path
_SRC = str(Path(".").resolve().parent / "src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1. Generate Synthetic Multi-Asset Returns

We simulate a 5-asset universe with correlated returns and a **regime change**:
- First half: calm market (low vol, positive drift)
- Second half: crisis regime (high vol, negative drift for risky assets)

This tests whether the RL agent can adapt its allocations dynamically.

In [ ]:
from environment import generate_synthetic_returns

ASSET_NAMES = ["Equities", "Bonds", "Gold", "Real Estate", "Cash"]
N_ASSETS = 5
N_STEPS = 500
SEED = 42

returns_data = generate_synthetic_returns(N_ASSETS, N_STEPS, seed=SEED)
print(f"Returns shape: {returns_data.shape}")
print("\nSummary statistics:")

summary = pd.DataFrame({
    "Asset": ASSET_NAMES,
    "Mean (daily)": np.mean(returns_data, axis=0),
    "Std (daily)": np.std(returns_data, axis=0),
    "Min": np.min(returns_data, axis=0),
    "Max": np.max(returns_data, axis=0),
})
summary.set_index("Asset", inplace=True)
summary

In [ ]:
# Visualise cumulative returns of individual assets
fig, ax = plt.subplots(figsize=(10, 5))
for i, name in enumerate(ASSET_NAMES):
    cum = np.cumprod(1 + returns_data[:, i])
    ax.plot(cum, label=name, linewidth=1.2)

ax.axvline(x=N_STEPS // 2, color="red", linestyle="--", alpha=0.5, label="Regime change")
ax.set_xlabel("Time Step")
ax.set_ylabel("Cumulative Return")
ax.set_title("Individual Asset Cumulative Returns")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Define Benchmarks

We compare the RL agent against three classical allocation strategies:

1. **Equal Weight (1/N)**: w_i = 1/N for all assets. Simple but hard to beat.
2. **Mean-Variance**: w* = Sigma^{-1} mu, clipped to long-only.
3. **Risk Parity**: w_i proportional to 1/sigma_i.

In [ ]:
from benchmarks import equal_weight, mean_variance, risk_parity, run_benchmark

ew_w = equal_weight(N_ASSETS)
mv_w = mean_variance(returns_data, risk_aversion=1.0)
rp_w = risk_parity(returns_data)

weights_df = pd.DataFrame({
    "Asset": ASSET_NAMES,
    "Equal Weight": ew_w,
    "Mean-Variance": mv_w,
    "Risk Parity": rp_w,
})
weights_df.set_index("Asset", inplace=True)
print("Benchmark Allocations:")
weights_df.round(4)

In [ ]:
# Run benchmarks
COST_RATE = 0.001

ew_result = run_benchmark(ew_w, returns_data, cost_rate=COST_RATE)
mv_result = run_benchmark(mv_w, returns_data, cost_rate=COST_RATE)
rp_result = run_benchmark(rp_w, returns_data, cost_rate=COST_RATE)

print(f"Equal Weight -- Return: {ew_result['total_return']:.4f}, "
      f"Sharpe: {ew_result['sharpe']:.4f}, CVaR: {ew_result['cvar_95']:.4f}")
print(f"Mean-Variance -- Return: {mv_result['total_return']:.4f}, "
      f"Sharpe: {mv_result['sharpe']:.4f}, CVaR: {mv_result['cvar_95']:.4f}")
print(f"Risk Parity -- Return: {rp_result['total_return']:.4f}, "
      f"Sharpe: {rp_result['sharpe']:.4f}, CVaR: {rp_result['cvar_95']:.4f}")

## 3. Train RL Agent with CVaR Constraint

The RL agent uses a feedforward policy network with **softmax output** to produce
valid portfolio weights. Training uses Natural Evolution Strategies (NES) with
mirror sampling -- a derivative-free approach since our network is pure numpy.

**Reward = portfolio_return - cvar_penalty - transaction_costs**

The CVaR penalty activates when the rolling tail risk exceeds a threshold,
encouraging the agent to manage downside risk.

In [ ]:
from agent import PolicyNetwork
from environment import PortfolioEnv
from trainer import run_episode, train_rl_agent

# Create environment and agent
env = PortfolioEnv(
    n_assets=N_ASSETS,
    lookback=10,
    initial_capital=1.0,
    cost_rate=COST_RATE,
    cvar_alpha=0.95,
    cvar_threshold=0.03,
)

agent = PolicyNetwork(
    state_dim=env.state_dim,
    action_dim=env.action_dim,
    hidden_dim=32,
    seed=SEED,
)

print(f"State dim: {env.state_dim}")
print(f"Action dim: {env.action_dim}")
print(f"Network parameters: {agent.num_params}")

In [ ]:
# Train the agent (this takes a few minutes)
train_results = train_rl_agent(
    agent=agent,
    env=env,
    returns_data=returns_data,
    n_epochs=100,
    population_size=40,
    sigma=0.02,
    lr=0.01,
    cvar_lambda=2.0,
    seed=SEED,
)

print(f"Final reward: {train_results['final_reward']:.6f}")

In [ ]:
# Plot training loss curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_results["loss_history"], linewidth=1.2, color="steelblue")
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (negative reward)")
ax.set_title("RL Agent Training Loss")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Loss decreased: {train_results['loss_history'][0]:.4f} -> "
      f"{train_results['loss_history'][-1]:.4f}")

## 4. Backtest All Strategies

Run the trained RL agent alongside all benchmarks on the same return data
and compare key metrics.

In [ ]:
# RL agent backtest
rl_result = run_episode(agent, env, returns_data)

# Build comparison table
comparison = pd.DataFrame([
    {
        "Strategy": "RL Agent",
        "Total Return": f"{rl_result['total_return']:.4f}",
        "Sharpe Ratio": f"{rl_result['sharpe']:.4f}",
        "CVaR (95%)": f"{rl_result['cvar']:.4f}",
        "Max Drawdown": f"{rl_result['max_drawdown']:.4f}",
    },
    {
        "Strategy": "Equal Weight",
        "Total Return": f"{ew_result['total_return']:.4f}",
        "Sharpe Ratio": f"{ew_result['sharpe']:.4f}",
        "CVaR (95%)": f"{ew_result['cvar_95']:.4f}",
        "Max Drawdown": f"{ew_result['max_drawdown']:.4f}",
    },
    {
        "Strategy": "Mean-Variance",
        "Total Return": f"{mv_result['total_return']:.4f}",
        "Sharpe Ratio": f"{mv_result['sharpe']:.4f}",
        "CVaR (95%)": f"{mv_result['cvar_95']:.4f}",
        "Max Drawdown": f"{mv_result['max_drawdown']:.4f}",
    },
    {
        "Strategy": "Risk Parity",
        "Total Return": f"{rp_result['total_return']:.4f}",
        "Sharpe Ratio": f"{rp_result['sharpe']:.4f}",
        "CVaR (95%)": f"{rp_result['cvar_95']:.4f}",
        "Max Drawdown": f"{rp_result['max_drawdown']:.4f}",
    },
])
comparison.set_index("Strategy", inplace=True)
comparison

## 5. Visualizations

In [ ]:
from diagnostics import (
    plot_allocation_evolution,
    plot_cumulative_returns,
    plot_drawdown_comparison,
    plot_risk_return_scatter,
)

In [ ]:
# Cumulative returns comparison
cum_returns = {
    "RL Agent": rl_result["cumulative_returns"],
    "Equal Weight": ew_result["cumulative_returns"],
    "Mean-Variance": mv_result["cumulative_returns"],
    "Risk Parity": rp_result["cumulative_returns"],
}

fig, ax = plot_cumulative_returns(cum_returns)
ax.axvline(x=N_STEPS // 2, color="red", linestyle="--", alpha=0.3, label="Regime change")
ax.legend(fontsize=8)
plt.show()

In [ ]:
# RL agent allocation evolution
fig, ax = plot_allocation_evolution(
    rl_result["weights_history"],
    ASSET_NAMES,
)
ax.axvline(
    x=(N_STEPS // 2) - 10,  # adjusted for lookback
    color="red", linestyle="--", alpha=0.5, label="Regime change",
)
ax.legend(fontsize=8, loc="upper left")
plt.show()

In [ ]:
# Risk-return scatter
scatter_df = pd.DataFrame([
    {"strategy": "RL Agent", "total_return": rl_result["total_return"], "cvar_95": rl_result["cvar"]},
    {"strategy": "Equal Weight", "total_return": ew_result["total_return"], "cvar_95": ew_result["cvar_95"]},
    {"strategy": "Mean-Variance", "total_return": mv_result["total_return"], "cvar_95": mv_result["cvar_95"]},
    {"strategy": "Risk Parity", "total_return": rp_result["total_return"], "cvar_95": rp_result["cvar_95"]},
])

fig, ax = plot_risk_return_scatter(scatter_df)
plt.show()

In [ ]:
# Drawdown comparison
# Compute RL drawdown
rl_cum = rl_result["cumulative_returns"]
rl_peak = np.maximum.accumulate(rl_cum)
rl_dd = (rl_peak - rl_cum) / np.maximum(rl_peak, 1e-12)

drawdowns = {
    "RL Agent": rl_dd,
    "Equal Weight": ew_result["drawdown"],
    "Mean-Variance": mv_result["drawdown"],
    "Risk Parity": rp_result["drawdown"],
}

fig, ax = plot_drawdown_comparison(drawdowns)
plt.show()

## 6. Interpretation: When Does RL Add Value?

### Key observations:

**Regime adaptability**: The RL agent can dynamically shift allocations in
response to changing market conditions. Static strategies (equal weight,
mean-variance, risk parity) use fixed weights determined from the full
sample, so they cannot adapt when a calm market transitions to crisis.

**CVaR management**: The reward function penalises the agent when rolling
CVaR exceeds the threshold (3%). This encourages the agent to reduce
exposure to volatile assets during turbulent periods, potentially at the
cost of some upside during calm periods.

**Transaction costs**: The RL agent incurs costs when rebalancing, while
static strategies pay only at inception. Frequent rebalancing can erode
returns, but the agent learns to balance adaptation against turnover.

**When RL adds value**:
- Regime changes and non-stationary markets where static allocations
  become suboptimal.
- When tail risk management matters more than raw returns.
- When the universe has assets with meaningfully different risk/return
  profiles that the agent can exploit.

**Limitations**:
- The NES training is computationally expensive (derivative-free).
- With limited training budget, the agent may not fully converge.
- The pure numpy implementation prioritises clarity over performance.
- In-sample evaluation (training and testing on same data) overstates
  real performance. A proper out-of-sample split would be needed.

**Extensions**: PyTorch-based policy gradient methods (PPO, SAC) would
converge faster and handle larger state/action spaces. Adding more
realistic market features (momentum, volatility clustering, macro
indicators) would improve the agent's decision making.